# Kaggle Autonomous Multi-Agent System (KAMAS)
## Competition: Playground Series - Season 6, Episode 9 (`playground-series-s6e9`)
### Electric Vehicle Adoption & Range Anxiety (Binary Classification)

This notebook provides a complete, leak-free, reproducible 5-fold GBDT pipeline.
It can be run directly inside Kaggle Notebooks attached to the competition dataset.

### 1. Environment & Repository Setup
Clones the GitHub repository or sets up the execution path.

In [ ]:
import os
import sys
from pathlib import Path

# If running in Kaggle notebook and repository is cloned:
REPO_NAME = "electric-vehicle"
if Path(f"/kaggle/working/{REPO_NAME}").exists():
    os.chdir(f"/kaggle/working/{REPO_NAME}")
    print(f"[*] Changed working directory to /kaggle/working/{REPO_NAME}")

# Add current directory to path
sys.path.insert(0, str(Path.cwd()))
print(f"[*] Current Working Directory: {Path.cwd()}")

### 2. Dependency Verification

In [ ]:
import numpy as np
import pandas as pd
import polars as pl
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

print(f"Polars version:   {pl.__version__}")
print(f"LightGBM version: {lgb.__version__}")
print(f"Pandas version:   {pd.__version__}")
print(f"NumPy version:    {np.__version__}")

### 3. Data Path Resolution & Ingestion
Automatically identifies `/kaggle/input/playground-series-s6e9` or local fallback.

In [ ]:
COMPETITION_ID = "playground-series-s6e9"

try:
    from kaggle.paths import resolve_data_dir
    data_dir = resolve_data_dir(competition_id=COMPETITION_ID)
except Exception:
    candidate_dirs = [
        Path(f"/kaggle/input/{COMPETITION_ID}"),
        Path(f"data/processed/{COMPETITION_ID}"),
        Path(f"data/raw/{COMPETITION_ID}"),
    ]
    data_dir = None
    for d in candidate_dirs:
        if d.exists() and ((d / "train.parquet").exists() or (d / "train.csv").exists()):
            data_dir = d
            break
    if data_dir is None and Path("/kaggle/input").exists():
        for f in Path("/kaggle/input").rglob("train.*"): 
            data_dir = f.parent
            break

print(f"[*] Ingesting datasets from: {data_dir}")

# Read Train
if (data_dir / "train.parquet").exists():
    train_df = pl.read_parquet(data_dir / "train.parquet").to_pandas()
else:
    train_df = pl.read_csv(data_dir / "train.csv").to_pandas()

# Read Test
if (data_dir / "test.parquet").exists():
    test_df = pl.read_parquet(data_dir / "test.parquet").to_pandas()
else:
    test_df = pl.read_csv(data_dir / "test.csv").to_pandas()

print(f"[+] Train dimensions: {train_df.shape}")
print(f"[+] Test dimensions:  {test_df.shape}")

### 4. Stratified 5-Fold Cross-Validation Setup
Certified StratifiedKFold ensuring exact balance of target prevalence across all 5 folds.

In [ ]:
id_col = "id"
target_col = "Will_Buy_EV"
features = [c for c in test_df.columns if c != id_col]

# Encode target to binary (0/1)
if train_df[target_col].dtype == object:
    y_train = (train_df[target_col] == "Yes").to_numpy().astype(int)
else:
    y_train = train_df[target_col].to_numpy().astype(int)

# Check for pre-computed certified folds
folds_path = data_dir / "folds.parquet"
if folds_path.exists():
    print(f"[*] Loading certified folds from {folds_path}")
    f_df = pl.read_parquet(folds_path).to_pandas()
    train_df = train_df.merge(f_df, on=id_col, how="left")
else:
    print("[*] Generating deterministic 5-fold StratifiedKFold (seed=42)...")
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    fold_arr = np.empty(len(train_df), dtype=np.int32)
    for f, (_, va) in enumerate(skf.split(train_df, y_train)):
        fold_arr[va] = f
    train_df["fold"] = fold_arr

# Convert categoricals to pandas 'category' dtype for native GBDT handling
cat_cols = [c for c in features if not pd.api.types.is_numeric_dtype(train_df[c])]
for c in cat_cols:
    train_df[c] = train_df[c].astype("category")
    test_df[c] = test_df[c].astype("category")

print(f"[*] Feature space ({len(features)}): {features}")
print(f"[*] Categorical features ({len(cat_cols)}): {cat_cols}")
print(f"[*] Target positive rate: {y_train.mean():.4f}")

### 5. 5-Fold LightGBM Model Training

In [ ]:
oof_preds = np.zeros(len(train_df), dtype=np.float64)
test_preds = np.zeros(len(test_df), dtype=np.float64)
fold_aucs = []
feature_importances = np.zeros(len(features), dtype=np.float64)

params = {
    "n_estimators": 1000,
    "learning_rate": 0.05,
    "num_leaves": 31,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_samples": 50,
    "verbose": -1,
    "n_jobs": -1,
}

print("[*] Training 5-Fold LightGBM Baseline...")
for fold in range(5):
    tr_mask = train_df["fold"] != fold
    va_mask = train_df["fold"] == fold

    X_tr, y_tr = train_df.loc[tr_mask, features], y_train[tr_mask]
    X_va, y_va = train_df.loc[va_mask, features], y_train[va_mask]

    model = lgb.LGBMClassifier(random_state=42 + fold, **params)
    model.fit(
        X_tr,
        y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[lgb.early_stopping(stopping_rounds=40, verbose=False)],
    )

    va_prob = model.predict_proba(X_va)[:, 1]
    te_prob = model.predict_proba(test_df[features])[:, 1]

    oof_preds[va_mask] = va_prob
    test_preds += te_prob / 5.0
    feature_importances += model.feature_importances_ / 5.0

    fold_auc = roc_auc_score(y_va, va_prob)
    fold_aucs.append(fold_auc)
    print(f"  Fold {fold+1} ROC-AUC: {fold_auc:.6f}")

overall_cv = roc_auc_score(y_train, oof_preds)
std_cv = float(np.std(fold_aucs))
print("=" * 60)
print(f"[+] Overall 5-Fold Out-of-Fold ROC-AUC: {overall_cv:.6f} (+/- {std_cv:.6f})")
print("=" * 60)

### 6. Submission Generation & Quality Gate 6 Verification

In [ ]:
output_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("experiments/submissions")
output_dir.mkdir(parents=True, exist_ok=True)
sub_path = output_dir / "submission.csv"

submission = pd.DataFrame({
    "id": test_df["id"],
    "Will_Buy_EV": test_preds,
})
submission.to_csv(sub_path, index=False)

# Quality Gate 6 Invariant Checklist
assert len(submission) == len(test_df), f"Row count mismatch: {len(submission)} vs {len(test_df)}"
assert list(submission.columns) == ["id", "Will_Buy_EV"], f"Headers incorrect: {list(submission.columns)}"
assert not submission["Will_Buy_EV"].isnull().any(), "Submission contains NaN/null values!"
assert (submission["Will_Buy_EV"] >= 0.0).all() and (submission["Will_Buy_EV"] <= 1.0).all(), "Values outside probability range!"

print(f"[+] Quality Gate 6 VERIFIED!")
print(f"[+] Saved submission to: {sub_path} ({sub_path.stat().st_size / (1024*1024):.2f} MB)")
print(submission.head(10))